# 08 — Limitations and Validation Roadmap

## Objective

This notebook defines the evidence boundaries of the SECOM Fail-risk
analysis and specifies the validation steps required before operational
use.

The objectives are:

1. distinguish supported conclusions from unsupported claims;
2. document data, evaluation, model, interpretation, and deployment
   limitations;
3. explain how these limitations affect engineering use;
4. propose a prioritized validation roadmap;
5. prevent retrospective predictive evidence from being interpreted as
   production readiness or physical causality.

This chapter does not introduce additional candidate models or optimize
the reported results.

## 8.1 Evidence Established by the Analysis

The completed analysis supports the following conclusions:

1. the anonymous SECOM measurements contain useful information for
   ranking Fail risk;
2. the Random Forest out-of-fold scores performed better than random
   ranking, with an Average Precision of 0.1627 compared with a class
   prevalence of approximately 0.0664;
3. the highest-risk 5% of samples had an observed Fail rate of 22.8%,
   approximately 3.43 times the overall Fail rate;
4. the highest-risk 20% of samples contained 51% of all observed Fail
   cases;
5. eight anonymous variables received repeated support from both
   validation-fold permutation importance and training-fold F-test
   selection;
6. five of the eight signals had agreement between their observed
   Pass/Fail direction and Logistic Regression coefficient direction;
7. risk scores, alert volumes, and candidate-signal flags changed
   substantially over time;
8. the model can support retrospective investigation prioritization, but
   it cannot replace existing testing and quality-control procedures.

## 8.2 Claims Not Established by the Analysis

The analysis does not establish that:

- the model scores are calibrated probabilities of physical failure;
- the selected Random Forest is universally superior to all alternative
  models;
- the reported performance will transfer unchanged to future production
  periods, products, tools, or fabrication facilities;
- the eight candidate signals are physical root causes;
- increasing or decreasing a candidate signal will causally change yield;
- the retrospective Red and Amber thresholds are permanent production
  specifications;
- Routine-tier samples are guaranteed to Pass;
- the model is ready for automatic product hold, release, rework, or scrap
  decisions.

## 8.3 Data Limitations

### Limited number of Fail samples

The dataset contains 1,567 samples but only 104 Fail cases. Under
ten-fold cross-validation, each validation fold contains approximately
ten or eleven Fail samples.

As a result:

- fold-level performance is sensitive to the ranking of a small number of
  Fail samples;
- Average Precision and feature importance vary across folds;
- rare Fail mechanisms may not be adequately represented;
- subgroup evaluation is severely limited.

### High dimensionality

The dataset contains 590 measurement variables, which is large relative
to the 104 observed Fail samples.

This increases the risk of:

- unstable feature selection;
- spurious associations;
- correlated or redundant signals;
- model dependence on sample-specific patterns.

### Missing measurements

Missing measurements are present across a large proportion of the feature
columns. Median imputation allowed the models to operate, but imputation
cannot recover the true unobserved measurement values.

The eight final candidate signals did not show reliable missingness
associations with Fail after multiple-testing adjustment. However, this
does not prove that missingness is irrelevant for all other variables or
future production data.

### Anonymous and incomplete manufacturing context

The measurement variables are anonymous. The dataset does not provide
sufficient information about:

- physical measurement definitions and units;
- product type;
- lot and wafer relationships;
- tool and chamber identity;
- recipe and process-step context;
- maintenance and equipment-event history;
- wafer-map coordinates.

Consequently, predictive signals cannot be mapped to physical mechanisms,
equipment excursions, or confirmed root causes.

## 8.4 Evaluation-Design Limitations

### Shuffled rather than temporal cross-validation

The main model evaluation used shuffled stratified ten-fold
cross-validation. This design made efficient use of the limited Fail
samples and maintained similar class proportions across folds.

However, it did not simulate strict past-to-future deployment because a
training fold could contain observations recorded later than its
validation observations.

The temporal monitoring analysis showed substantial changes in:

- median risk score;
- Red and Amber alert rates;
- observed Fail rate;
- candidate-signal flag frequency.

This indicates that strict time-based validation is necessary before
operational use.

### No independent external validation dataset

All model development, comparison, signal analysis, and retrospective
monitoring were performed using the same SECOM dataset.

Out-of-fold prediction reduced training-set optimism, but it did not
replace validation on a fully independent production period, product, or
fabrication environment.

### Model selection and evaluation on the same dataset

Candidate models were compared using the same cross-validation framework.
The selected Random Forest was then used for downstream signal and
monitoring analysis.

Although the out-of-fold predictions remained leakage-safe, selecting a
model after comparing multiple candidates may still introduce some
selection optimism. Independent confirmation would therefore be required
before deployment.

## 8.5 Model-Performance Limitations

The Random Forest achieved:

- out-of-fold Average Precision: 0.1627;
- out-of-fold ROC-AUC: 0.7457.

These results indicate useful but limited risk-ranking ability.

The highest-risk 5% of samples had an observed Fail rate of 22.8%.
Therefore, approximately 77.2% of the Red-tier samples were still Pass.

The highest-risk 20% contained 51% of all Fail samples, but the remaining
49% of Fail samples were assigned to the Routine tier.

These results support investigation prioritization, but they are not
sufficient for automatic classification or product disposition.

### Uncalibrated model scores

The Random Forest score range was approximately 0.0634 to 0.4707, and no
sample reached the conventional 0.5 classification threshold.

The scores were therefore interpreted as relative risk-ranking values
rather than calibrated probabilities.

For example, a score of 0.40 should not be interpreted as a confirmed 40%
physical probability of failure.

### Modest difference between candidate models

Random Forest was selected as the primary downstream model, but its
Average Precision was only modestly higher than that of Logistic
Regression.

The selection was based on a combination of:

- useful out-of-fold risk ranking;
- nonlinear and interaction capability;
- practical threshold and workload trade-offs;
- compatibility with validation-fold permutation importance.

The result should not be interpreted as evidence that Random Forest is
universally or significantly superior to all alternatives.

## 8.6 Candidate-Signal Interpretation Limitations

### Predictive importance is not physical causality

Permutation importance measures whether model performance decreases when
a feature is shuffled. F-test selection measures univariate statistical
separation. Logistic coefficients describe conditional model
associations.

None of these methods proves that changing a signal will physically cause
a change in yield.

### Correlated features may share or exchange importance

When multiple measurements contain similar information, the Random Forest
may use one feature in place of another. Shuffling one correlated feature
may therefore produce a small importance value even when the underlying
measurement family is relevant.

Similarly, Logistic Regression coefficient magnitudes and directions may
change when correlated variables are included together.

### Direction disagreement

Five consensus signals showed agreement between observed Pass/Fail median
direction and Logistic Regression coefficient direction.

Feature_163, feature_79, and feature_122 showed disagreement between the
two forms of directional evidence. These variables were retained as
context-dependent predictive signals but were not translated into simple
high-value or low-value alarm rules.

### Global signals are not local explanations

The eight consensus variables were selected because they showed stable
evidence across folds and methods.

They do not fully explain why every individual sample received a
particular Random Forest risk score because:

- the Random Forest used all 590 variables;
- prediction may depend on nonlinear relationships;
- interactions may be sample-specific;
- the percentile flags are descriptive reference markers rather than
  local model-contribution estimates.

## 8.7 Monitoring-Prototype Limitations

### Retrospective thresholds

The Red and Amber thresholds were derived from the distribution of
out-of-fold scores in the current dataset.

They are retrospective prototype thresholds rather than permanent
production specifications.

### Unstable alert volume over time

The same global thresholds produced substantially different alert rates
across calendar weeks.

Some early weeks assigned most samples to the Red or Amber queue, while
several later weeks produced few or no priority alerts.

This would create unstable engineering workload and indicates that score
distribution and alert-volume monitoring are required.

### No observed operational cost data

The dataset does not provide:

- cost of investigating a false alarm;
- cost of missing a Fail sample;
- available engineering capacity;
- delay caused by additional review;
- cost of product hold, rework, or scrap.

Therefore, the project reports multiple investigation-budget scenarios
rather than claiming a single economically optimal threshold.

## 8.8 Limitation and Mitigation Summary

| Limitation | Potential impact | Recommended mitigation |
|---|---|---|
| Only 104 Fail samples | Unstable fold metrics and limited subgroup analysis | Collect more labeled Fail cases and report uncertainty |
| 590 anonymous features | Limited physical interpretation | Map variables to measurement definitions and process steps |
| Missing measurements | Imputation uncertainty | Monitor missingness and investigate measurement-system causes |
| Shuffled cross-validation | Does not simulate future deployment | Add strict past-to-future validation |
| No external dataset | Unknown transferability | Validate on a later period, product, or fabrication environment |
| Uncalibrated scores | Scores cannot be read as true probabilities | Perform calibration only after independent validation |
| Correlated measurements | Importance and coefficient instability | Review correlated signal groups with process engineers |
| Anonymous signal importance | No causal root-cause conclusion | Combine with engineering experiments and process metadata |
| Retrospective thresholds | Unstable future alert volume | Run shadow mode and monitor score distributions |
| No investigation-cost data | No proven optimal threshold | Measure workload, miss cost, and alert value during shadow mode |

## 8.9 Prioritized Validation Roadmap

### Priority 1 — Restore manufacturing context

Obtain and link:

- measurement names and units;
- lot and wafer identifiers;
- product type;
- tool and chamber;
- recipe and process step;
- maintenance and equipment-event history.

This is required before the candidate signals can be translated into
physical engineering hypotheses.

### Priority 2 — Perform temporal validation

Train the fixed preprocessing and Random Forest pipeline using an earlier
time period and evaluate it on a strictly later period.

The evaluation should examine:

- Average Precision and ROC-AUC;
- Fail recall and precision;
- score-distribution shift;
- threshold transfer;
- alert-volume stability;
- candidate-signal stability.

### Priority 3 — Run shadow-mode monitoring

Operate the model without changing product disposition.

Log:

- risk scores and tiers;
- alert volume;
- candidate-signal context;
- final outcomes;
- engineering review decisions;
- confirmed issues;
- investigation time and workload.

### Priority 4 — Confirm signal meaning

Process engineers should review the candidate signals and determine:

- their physical measurement definitions;
- relevant process steps and equipment;
- known control limits;
- possible confounding factors;
- whether abnormal values are repeatable;
- whether controlled experiments are feasible.

### Priority 5 — Reassess thresholds and calibration

Only after collecting independent shadow-mode data should the project:

- select an operating threshold based on investigation capacity;
- estimate false-alarm and missed-Fail costs;
- evaluate probability calibration;
- revise risk tiers;
- decide whether retraining is required.

## 8.10 Deployment Risk Register

| Risk | Current evidence | Operational decision |
|---|---|---|
| False alarms | Red precision was 22.8% | Do not automatically hold or scrap products |
| Missed Fail cases | Routine contained 49% of Fail samples | Preserve existing QC and audit Routine samples |
| Temporal drift | Weekly score and alert distributions changed substantially | Monitor drift and perform time-based validation |
| Unstable workload | Weekly priority-queue rates varied widely | Define capacity limits and escalation rules |
| Incorrect causal interpretation | Features are anonymous and observational | Use “candidate predictive signal,” not “root cause” |
| Data-quality failure | Original data contain widespread missingness | Add schema and missingness checks before scoring |
| Threshold transfer failure | Thresholds came from retrospective OOF scores | Re-estimate and validate thresholds in shadow mode |

## 8.11 Conclusions

The SECOM analysis established useful retrospective Fail-risk ranking,
risk-tier enrichment, and a stable set of anonymous candidate signals.

However, the evidence does not establish calibrated failure probability,
physical causality, future-period generalization, or production readiness.

The most important remaining validation requirement is strict
past-to-future evaluation because the retrospective monitoring analysis
showed substantial temporal changes in score, alert, and signal
distributions.

A real implementation should first restore manufacturing context and run
the model in shadow mode. Existing testing and quality-control procedures
should remain unchanged while alert workload, final outcomes, data drift,
and engineering investigation value are measured.

The current result should therefore be interpreted as a reproducible
retrospective engineering decision-support prototype and a foundation for
future validation, not as an automatic product-disposition system.